In [2]:
import pandas as pd
import os
from google.cloud import bigquery
from dotenv import load_dotenv

# 1. Cargar la variable de entorno que apunta a tu gcp_key.json
# Asegúrate de que en tu archivo .env existe la línea: 
# GOOGLE_APPLICATION_CREDENTIALS="tu_llave.json"
load_dotenv('../.env')

# 2. FIX DE SEGURIDAD: Inyectar la ruta exacta de la llave a las variables de entorno de tu computadora
# El '../' le dice a Python: "Sal de la carpeta notebooks y busca el archivo JSON en la carpeta principal"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../gcp_key.json'

# Inicializar el cliente oficial de BigQuery
client = bigquery.Client()

print("Descargando datos crudos desde BigQuery...")

# 2. Extraer los datos crudos
# Fíjate en el uso de acentos graves (backticks `) alrededor de la ruta de la tabla
query = "SELECT * FROM `mktng-performance-dashboard.data.raw`;"

# Ejecutamos la consulta y la convertimos directamente en un DataFrame de Pandas
df = client.query(query).to_dataframe()

print("¡Datos descargados! Calculando KPIs...")

# 3. Calcular los KPIs de Marketing 
df['ctr'] = (df['clicks'] / df['impressions']).fillna(0)
df['cpc'] = (df['mark_spent'] / df['clicks']).fillna(0)
df['cpa'] = (df['mark_spent'] / df['orders']).fillna(0)
df['roas'] = (df['revenue'] / df['mark_spent']).fillna(0)

# Mostrar una vista previa
print(df[['campaign_id', 'c_date', 'ctr', 'cpc', 'cpa', 'roas']].head())

# 4. Guardar los resultados
# Guardamos copia local en CSV por si acaso (buena práctica de respaldo)
df.to_csv('../data/processed/metrics_clean.csv', index=False)

# 5. Cargar de vuelta a la nube usando el cliente oficial de Google
# En BigQuery siempre es más seguro usar la ruta completa: proyecto.dataset.tabla
tabla_destino_completa = 'mktng-performance-dashboard.data.marketing_kpis'

print("\nSubiendo datos procesados a Google BigQuery...")

# Le decimos a Google que si la tabla ya existe, la sobreescriba (equivalente a 'replace')
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE" 
)

# Usamos el cliente para subir el DataFrame directamente
job = client.load_table_from_dataframe(
    df, 
    tabla_destino_completa, 
    job_config=job_config
)

# Esperamos a que el trabajo en la nube termine
job.result()

print(f"✅ ¡Métricas calculadas y tabla '{tabla_destino_completa}' creada en la nube con éxito!")

Descargando datos crudos desde BigQuery...


C:\Users\jbav9\OneDrive\Desktop\Portafolio\Digital Marketing Performance Dashboard\marketing-performance-dashboard\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


¡Datos descargados! Calculando KPIs...
   campaign_id      c_date       ctr        cpc          cpa      roas
0       374754  2021-02-01  0.012438  15.757076  3372.014286  1.655687
1       374754  2021-02-02      0.01   4.645895       4639.7  1.039291
2       374754  2021-02-03   0.01348   21.60249  3810.509804  1.307174
3       374754  2021-02-04      0.01  12.447178  3928.394737  1.504177
4       374754  2021-02-05  0.006967   2.428294    4553.9625  1.312483

Subiendo datos procesados a Google BigQuery...
✅ ¡Métricas calculadas y tabla 'mktng-performance-dashboard.data.marketing_kpis' creada en la nube con éxito!
